# Step 21 — Monthly phase-file indexing

The workflow is implemented directly in this notebook. Shared parsing/filter helpers remain in `pinatubo_steps.pha_parser`; step 20 also uses FLOVOpy's VDAP reader. This removes the redundant step-specific module while retaining a single shared implementation of the underlying parsing rules.

Legacy PHA files are read without modification. Running the processing cell replaces the configured pick-index CSV and error log beneath the external output root. These are arrival picks and source groupings, not relocated earthquakes or confirmed waveform associations. No clock offset/drift correction is applied. UTC-style timestamps reflect parser representation, not independently verified historical timing.

Run both steps 20 and 21 before step 22, which applies the cross-source merge/priority rules. Review error logs and grouping assumptions before merging.

## 1. Load and review configuration

Choose `CONFIG_FILE` if needed. Inspect source and destination paths and filter settings before executing the processing cell. This cell only resolves settings.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    pha_dir=P['LEGACY_PHA_MONTHLY_DIR'],
    out_csv=P['MONTHLY_PICK_INDEX'],
    error_log=P['MONTHLY_PICK_ERR'],
    max_pick_span=30.0,
)
display(SETTINGS)

## 2. Reader and workflow implementation

Monthly inputs are searched in the configured directory, without recursive traversal, using `.PHA` and `.pha` extensions. The shared Pinatubo parser preserves separator-defined event blocks and handles the legacy fixed-width/tokenized line conventions. This is a different input dialect from the individual-event reader.

`event_id` is the source stem plus a zero-based block number; it is a source grouping key, not a hypocenter or verified waveform match. `max_pick_span=30` is passed to the shared group filter. That filter deduplicates and applies a median-centered ±30-second window; its repeated span check does not guarantee a final total duration of 30 seconds. Fewer than three picks bypass the median outlier test. Preserve this distinction when interpreting the setting.

The workflow counts retained, trimmed and dropped blocks. Original line text/number and source filename accompany each emitted pick, along with mapped ID, phase, timing, onset, polarity and weight. Pick counters can change when the input inventory changes. Logged parser issues can coexist with an output CSV: a completed run is not evidence that every legacy line was understood.

The CSV and log are written at completion. If no valid rows remain, execution stops and an older output can still exist; review the current run rather than relying on file existence alone.

The next cell defines functions only. It does not read data or write products.

In [ ]:
from pathlib import Path

import pandas as pd

from pinatubo_steps.pha_parser import (
    parse_pha_file,
    filter_pick_group,   # NEW — imported from refactored pha_parser.py
)

def run(
    *,
    pha_dir,
    out_csv,
    error_log,
    max_pick_span=30.0
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        pha_dir=pha_dir,
        out_csv=out_csv,
        error_log=error_log,
        max_pick_span=max_pick_span,
    )

    pha_dir = Path(args.pha_dir)
    out_csv = Path(args.out_csv)
    error_log = Path(args.error_log)

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    error_log.parent.mkdir(parents=True, exist_ok=True)

    pha_files = sorted(
        path for pattern in ("*.PHA", "*.pha") for path in pha_dir.glob(pattern)
    )
    if not pha_files:
        raise SystemExit(f"No .PHA files found in {pha_dir}")

    rows = []
    errors = []

    print(f"Found {len(pha_files)} monthly PHA files")

    kept_blocks = 0
    dropped_blocks = 0
    trimmed_blocks = 0

    pick_counter = 0

    for pha_file in pha_files:
        events = parse_pha_file(pha_file, errors)

        for block_idx, ev in enumerate(events):
            event_id = f"{pha_file.stem}_block{block_idx:04d}"

            raw_picks = ev.get("picks", [])
            if not raw_picks:
                dropped_blocks += 1
                continue

            # -------------------------------------------------------------
            # Sanity filter picks within the block
            # -------------------------------------------------------------
            filtered_picks = filter_pick_group(
                raw_picks,
                max_span_seconds=args.max_pick_span,
            )

            if not filtered_picks:
                dropped_blocks += 1
                errors.append(
                    f"{pha_file.name}:{event_id}: all picks rejected as outliers"
                )
                continue

            if len(filtered_picks) < len(raw_picks):
                trimmed_blocks += 1

            kept_blocks += 1

            # -------------------------------------------------------------
            # Emit rows
            # -------------------------------------------------------------
            for p in filtered_picks:
                pick_counter += 1

                rows.append({
                    "raw_line": p.get("raw_line"),
                    "raw_lineno": p.get("raw_lineno"),                    
                    "event_id": event_id,
                    "event_id_source": "monthly_phase_block",
                    "pick_id": f"monpha_{event_id}_{pick_counter}",
                    "pha_file": pha_file.name,
                    "station": p.get("station"),
                    "channel": p.get("channel"),
                    "seed_id": p.get("seed_id"),
                    "phase": p.get("phase"),
                    "pick_time": str(p.get("time")),
                    "onset": p.get("onset"),
                    "first_motion": p.get("first_motion"),
                    "weight": p.get("weight"),
                })


    if not rows:
        raise SystemExit("No valid monthly picks produced")

    df = pd.DataFrame(rows)
    df.sort_values(
        ["event_id", "station", "pick_time"],
        inplace=True,
    )

    df.to_csv(out_csv, index=False)

    with open(error_log, "w") as f:
        for e in errors:
            f.write(e + "\n")

    print("\n=== STEP 21 SUMMARY ===")
    print(f"Monthly PHA files:      {len(pha_files)}")
    print(f"Blocks kept:            {kept_blocks}")
    print(f"Blocks trimmed:         {trimmed_blocks}")
    print(f"Blocks dropped:         {dropped_blocks}")
    print(f"Total picks written:    {len(df)}")
    print(f"CSV written:            {out_csv}")
    print(f"Errors logged:          {error_log}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## 3. Run the step

This cell performs parsing and writes the configured CSV and log. Inputs remain unchanged. Review the printed summary and any exceptions before continuing.

In [ ]:
result = run(**SETTINGS)

## 4. Inspect results and error log

Preview the pick table and recorded issues. For monthly sources, pay particular attention to trimmed/dropped blocks. Then run the other phase-indexing notebook and continue to step 22; do not infer waveform matches from these source IDs alone.

In [ ]:
display(result['df'].head())
print('Output rows:', len(result['df']))
print('Source groups:', result['df']['event_id'].nunique())
log = Path(SETTINGS['error_log'])
print('Error log:', log)
print(log.read_text() if log.exists() and log.stat().st_size else 'No logged issues in this completed run.')